In [0]:
storage_account_name = "awcriptoprojectstoracc"
storage_account_key = dbutils.secrets.get(scope = 'crypto-data-key', key = 'storagekey').strip()

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
stream_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/cripto-stream-ns-01/cripto-live-trades/"

df_avro = (spark.read.format("avro")
           .option("recursiveFileLookup", "true")
           .option("ignoreExtension", "true")
           .load(stream_path))

display(df_avro.limit(10))

In [0]:
from pyspark.sql.functions import col, decode, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

df_final_clean = df_avro.withColumn(
    "trade_json", 
    decode(col("Body"), "UTF-8")
).select(
    col("SequenceNumber"),
    col("Offset"),
    col("EnqueuedTimeUtc"),
    col("trade_json")
)

display(df_final_clean.limit(10))

In [0]:

df_final_clean.write\
    .format("delta")\
        .mode("append")\
            .saveAsTable("Crypto_project_cat.default.streaming_crypto_data")